In [0]:
%sql

-- Drop the schema and all its tables/assets
DROP SCHEMA IF EXISTS dbr_dev.landing CASCADE;

-- Recreate the empty schema
CREATE SCHEMA IF NOT EXISTS dbr_dev.landing;

In [0]:
%sql
CREATE schema IF NOT EXISTS dbr_dev.joshuandegwa_gold;


In [0]:
%sql
use catalog dbr_dev;
show volumes;

In [0]:
%sql


-- select * from dbr_dev.joshuandegwa_bronze.claim_images;

-- select * from dbr_dev.joshuandegwa_bronze.claims limit 5;
-- select * from dbr_dev.joshuandegwa_bronze.customers limit 5;-- ok
-- select * from dbr_dev.joshuandegwa_bronze.policies limit 5;-- ok
-- select * from dbr_dev.joshuandegwa_bronze.telemetry limit 5; -- ok
-- select * from dbr_dev.joshuandegwa_bronze.training_images limit 5; -- 0
-- select * from dbr_dev.joshuandegwa_bronze.training_metadata limit 5; -- 0


In [0]:
import os

# Create the directory inside the volume path if it doesn't exist
dbutils.fs.mkdirs("/Volumes/dbr_dev/landing/_schemas/telemetry")
dbutils.fs.mkdirs("/Volumes/dbr_dev/landing/_checkpoints/telemetry")

In [0]:
%sql

-- INSERT INTO customers (customer_id, date_of_birth, borough, neighborhood, zip_code, name)
-- VALUES (105.0, '1985-06-15', 'MANHATTAN', 'UPPER EAST SIDE', '10021', 'Schmitt, Marlene');
select * from dbr_dev.joshuandegwa_bronze.customers where customer_id = '105.0';


In [0]:
%sql
-- # DELETE FROM claims 
-- # WHERE claim_no = 'f59f665f-c82f-4aa3-98b5-060c7a8f6d19';
select * from dbr_dev.joshuandegwa_bronze.claims WHERE claim_no = 'f59f665f-c82f-4aa3-98b5-060c7a8f6d19';

-- UPDATE policies 
-- SET chassis_no = 'VF1BZBCT7FR999999', 
--     premium = 1550.0 
-- WHERE policy_no = '102122649';
select * from dbr_dev.joshuandegwa_bronze.policies where policy_no = '102122649';

In [0]:
print("Spark Version:", spark.version)

In [0]:
from pyspark.sql.functions import col, current_timestamp, from_json
from pyspark.sql.types import DoubleType, StringType, StructField, StructType

# 1. Confluent Cloud Connection Parameters
CONFLUENT_BOOTSTRAP = "pkc-921jm.us-east-2.aws.confluent.cloud:9092"
CONFLUENT_API_KEY = "443GI6WDMGYIYHYD"
CONFLUENT_API_SECRET = (
    "cfltvgM+Ukf0dlkXLSyOyxLOFB6mclAITEyKW4OJjofB64y2HKCw3z1gYQYKStyQ"
)
TOPIC_NAME = "telemetry-stream"

# 2. Define the schema matching your telemetry data
telemetry_schema = StructType([
    StructField("chassis_no", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("speed", DoubleType(), True),
])

# 3. Paths using your Unity Catalog structure
CHECKPOINT_PATH = "/Volumes/dbr_dev/landing/telemetry_checkpoints/kafka_landing/"
OUTPUT_TABLE = "dbr_dev.landing.telemetry_landing"

# 4. Read stream using flattened kafka options for serverless compatibility
kafka_stream_df = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", CONFLUENT_BOOTSTRAP)
    .option("subscribe", TOPIC_NAME)
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option(
        "kafka.sasl.jaas.config",
        "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required"
        f' username="{CONFLUENT_API_KEY}" password="{CONFLUENT_API_SECRET}";',
    )
    .load()
)

# 5. Alternative explicit options if JAAS still throws classloader errors:
# (If the above still fails, swap the .option("kafka.sasl.jaas.config", ...) block with:)
# .option("kafka.sasl.username", CONFLUENT_API_KEY)
# .option("kafka.sasl.password", CONFLUENT_API_SECRET)

# 6. Parse JSON value and add your standard metadata enrichment columns
df_enriched = (
    kafka_stream_df.select(
        from_json(col("value").cast("string"), telemetry_schema).alias("data"),
        col("timestamp").alias("kafka_ingest_time"),
    )
    .select("data.*", "kafka_ingest_time")
    .withColumn("ingestion_time", current_timestamp())
)

# 7. Write stream to Unity Catalog table
query = (
    df_enriched.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .toTable(OUTPUT_TABLE)
)

query.awaitTermination()
print(
    f"🚀 Successfully streamed messages from Confluent into {OUTPUT_TABLE}!"
)

In [0]:
%sql
SELECT * FROM dbr_dev.joshuandegwa_bronze.telemetry_landing order by ingestion_time desc LIMIT 20;

In [0]:
result = spark.sql("SELECT COUNT(*) as total_rows FROM dbr_dev.landing.telemetry_landing")
display(result)

In [0]:
%sql
DROP TABLE IF EXISTS dbr_dev.joshuandegwa_bronze.telemetry_landing;

In [0]:
dbutils.fs.rm("/Volumes/dbr_dev/landing/telemetry_checkpoints/kafka_landing/", True)
print("Stale checkpoint cleared")

In [0]:
# Test listing the root of your Cloudflare R2 bucket
display(dbutils.fs.ls("r2://smart-claims-bucket@8d6974c4eec2dd94eaeb74af884b60d9.r2.cloudflarestorage.com/"))

In [0]:
cloud_file_path = "r2://smart-claims-bucket@8d6974c4eec2dd94eaeb74af884b60d9.r2.cloudflarestorage.com/claims/images/"
checkpoint_path = "r2://smart-claims-bucket@8d6974c4eec2dd94eaeb74af884b60d9.r2.cloudflarestorage.com/_checkpoints/claims_images"

df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "binaryFile")
    .load(cloud_file_path)
)

display(df, checkpointLocation=checkpoint_path)

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS dbr_dev.landing.archive